# Week 1 build · Gradients by hand, then by machine

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-01.ipynb)

**~3 hours · CPU only · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

| Part | Topic | Time |
|---|---|---|
| A | Shape drills and one silent bug | 25 min |
| B | A vOICe-style image→sound matrix | 35 min |
| C | Nudge-and-measure vs autograd | 20 min |
| D | Day 4's graph by hand, checked three ways | 30 min |
| E | Fit a line by gradient descent, then break it | 45 min |
| F | Stretch: a tiny autograd engine | 25 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Part A · Shape drills (25 min)

**Words used in this part.** *shape*: size along each axis. *broadcasting*: line shapes up from the right; sizes must be equal, or one of them 1 or missing. *batch*: examples stacked along the first axis.

In [ ]:
#@title 🔮 Predict A1: result shapes of the three operations below (or 'error')
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
# 1) rand(32,4,4) * rand(4,4)     2) rand(3,1) + rand(1,4)     3) rand(32,4,4) * rand(32)
predict("A1", prediction, confidence)

In [ ]:
ops = {
  "1) (32,4,4) * (4,4)": lambda: torch.rand(32, 4, 4) * torch.rand(4, 4),
  "2) (3,1) + (1,4)":    lambda: torch.rand(3, 1) + torch.rand(1, 4),
  "3) (32,4,4) * (32,)": lambda: torch.rand(32, 4, 4) * torch.rand(32),
}
for name, op in ops.items():
    try:
        print(name, "->", tuple(op().shape))
    except RuntimeError as e:
        print(name, "-> ERROR:", e)

**Repair it.** Operation 3 meant "one intensity per frame". The next cell **fails on purpose** until you fix it: change only the *shape* of the second tensor so the result is `(32, 4, 4)`.

In [ ]:
frames, per_frame = torch.rand(32, 4, 4), torch.rand(32)
fixed = frames * per_frame      # <- edit this line (hint: .reshape or [:, None, None])
print(fixed.shape)

In [ ]:
#@title 🔮 Predict A2: pred is (100,1), target is (100,), nearly equal values. What does ((pred-target)**2).mean() give: ~0, or something large? Why?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A2", prediction, confidence)

In [ ]:
target = torch.rand(100)
pred = (target + 0.01 * torch.randn(100)).unsqueeze(1)      # shape (100, 1): what a model usually outputs
buggy = ((pred - target) ** 2).mean()
right = ((pred.squeeze(1) - target) ** 2).mean()
print("pred", tuple(pred.shape), "target", tuple(target.shape), "-> (pred - target) is", tuple((pred - target).shape))
print(f"buggy loss {buggy:.4f}    correct loss {right:.6f}")

In [ ]:
#@title ✍️ A2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A2", what_happened, surprise)

---
## Part B · A sensory-substitution device is a matrix (35 min)

**Words used in this part.** *matmul* (`@`): each output is one row of the matrix dotted with the input. *linear map*: outputs are weighted sums of inputs, nothing else. *squash*: a matrix whose columns all point the same way collapses different inputs onto the same output.

The vOICe scans an image left to right. For each image **column**, every pixel **row** owns a sine wave (top = high pitch); brightness sets loudness; the ear gets the sum. For one column of 16 brightness values `x`, the audio snippet is `W @ x`, where column `j` of `W` is row `j`'s sine wave. We build `W` by hand. In week 12 you will *learn* one.

In [ ]:
sr, n_rows, col_dur = 8000, 16, 0.06                      # sample rate, image rows, seconds per image column
t = torch.arange(int(sr * col_dur)) / sr                   # time axis of one column's snippet
freqs = torch.logspace(np.log10(2800), np.log10(350), n_rows)   # row 0 (top) = highest pitch
W = torch.sin(2 * np.pi * freqs[None, :] * t[:, None])    # one sine wave per image row
print("t", tuple(t.shape), "freqs", tuple(freqs.shape), "W", tuple(W.shape))

In [ ]:
#@title 🔮 Predict B1: img is (16,16). What is the shape of W @ img, and which of its axes is 'time within a snippet'?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
img = torch.eye(16).flip(0)                 # a rising diagonal line: bottom-left to top-right
snips = W @ img                             # every image column sonified at once
audio = snips.T.reshape(-1)                 # put the columns one after another in time
print("W @ img:", tuple(snips.shape), "-> audio:", tuple(audio.shape), f"= {len(audio)/sr:.2f} s")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("image")
ax[1].specgram(audio.numpy(), Fs=sr, NFFT=256, noverlap=128); ax[1].set_title("spectrogram of the sound"); ax[1].set_ylim(0, 3200)
plt.show(); display(Audio(audio.numpy(), rate=sr))

Listen, and compare the spectrogram with the image: the matrix has turned *row* into *pitch* and *column* into *time*. Try `img = torch.eye(16)` (falling line) or draw your own 16×16 pattern.

In [ ]:
#@title 🔮 Predict B2: Squash W so every image row uses the SAME sine wave. Will a rising line and a falling line still sound different? Why / why not?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B2", prediction, confidence)

In [ ]:
W_squashed = W[:, [8]].repeat(1, n_rows)          # every column of W is now the same wave
rising, falling = torch.eye(16).flip(0), torch.eye(16)
for name, M in [("original W", W), ("squashed W", W_squashed)]:
    a, b = (M @ rising).T.reshape(-1), (M @ falling).T.reshape(-1)
    print(f"{name}: difference between rising and falling sound = {(a - b).abs().max():.4f}")
display(Audio((W_squashed @ rising).T.reshape(-1).numpy(), rate=sr))

In [ ]:
#@title ✍️ B2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B2", what_happened, surprise)

That is day 2's "squash" in a device you know: once two inputs land on the same output, **no later stage can separate them**, including the listener's brain. Keep this picture for week 8 (low rank, LoRA).

---
## Part C · Nudge-and-measure vs autograd (20 min)

**Words used in this part.** *derivative*: output change per tiny input change, at one point. *numerical gradient*: estimate it by nudging by `h`. *autograd*: PyTorch records the graph and applies the chain rule for you.

We use `f(x) = x³ − 3x` at `x = 2`. The exact derivative is `3x² − 3 = 9`.

In [ ]:
#@title 🔮 Predict C1: Which nudge size h gives the MOST accurate estimate with 32-bit floats: 1e-1, 1e-3, or 1e-8? What goes wrong with the others?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
f = lambda x: x**3 - 3*x
x = torch.tensor(2.0)                                   # 32-bit float, like model weights
for h in [1e-1, 1e-2, 1e-3, 1e-4, 1e-6, 1e-8]:
    est = (f(x + h) - f(x)) / h
    print(f"h = {h:<7g} estimate = {est.item():<12.5f} error = {abs(est.item() - 9):.5f}")

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

Two errors fight: a big `h` is not "local" enough; a tiny `h` drowns in rounding (at `1e-8`, `x + h == x` in 32-bit). This is one more reason nobody trains with numerical gradients. Now the real tool:

In [ ]:
x = torch.tensor(2.0, requires_grad=True)   # "track what happens to me"
y = f(x)                                    # forward pass: graph recorded
y.backward()                                # backward pass: chain rule
print("autograd says", x.grad.item())

---
## Part D · Day 4's graph by hand (30 min)

**Words used in this part.** *forward pass*: compute values. *backward pass*: walk from the output back, `gradient of my input = local derivative × gradient of my output`. *Add copies, multiply swaps.*

On paper, for `L = (a·b + c)²` with **a = 1.5, b = 4, c = −2**: compute `d`, `e`, `L`, then `∂L/∂e`, `∂L/∂c`, `∂L/∂a`, `∂L/∂b`. Then type your answers in.

In [ ]:
#@title 🔮 Predict D1: your hand-computed gradients
my_dL_da = 0.0  #@param {type:"number"}
my_dL_db = 0.0  #@param {type:"number"}
my_dL_dc = 0.0  #@param {type:"number"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", f"dL/da={my_dL_da}, dL/db={my_dL_db}, dL/dc={my_dL_dc}", confidence)

In [ ]:
def L_fn(a, b, c): return (a * b + c) ** 2
vals = dict(a=1.5, b=4.0, c=-2.0)

p = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in vals.items()}
L_fn(**p).backward()                                                  # 1) autograd
h = 1e-6
for k, mine in zip("abc", [my_dL_da, my_dL_db, my_dL_dc]):
    bumped = dict(vals); bumped[k] += h
    numeric = (L_fn(**bumped) - L_fn(**vals)) / h                     # 2) nudge-and-measure
    auto = p[k].grad.item()
    mark = "OK " if abs(mine - auto) < 1e-3 else "XX "
    print(f"{mark} dL/d{k}: yours {mine:8.3f} | autograd {auto:8.3f} | numerical {numeric:8.3f}")

In [ ]:
#@title 🔮 Predict D2: New graph: L = (a·b + a)², same a and b. 'a' is now used TWICE. What is dL/da? (Hint: two paths.)
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", prediction, confidence)

In [ ]:
a = torch.tensor(1.5, requires_grad=True); b = torch.tensor(4.0, requires_grad=True)
L = (a * b + a) ** 2
L.backward()
e = (a * b + a).item()
print(f"e = {e}, dL/de = {2*e}")
print(f"path through a*b: {2*e} x b = {2*e*4.0}    path through +a: {2*e} x 1 = {2*e}")
print("autograd dL/da =", a.grad.item(), " <- the two paths ADD")

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

---
## Part E · Fit a line by gradient descent, then break it (45 min)

**Words used in this part.** *model*: here `pred = w·x + b`, two parameters. *loss*: one number for how wrong we are; here **MSE**, the mean of `(pred − y)²`. *gradient descent*: `param ← param − learning_rate × gradient`. *diverge*: the loss explodes.

Data: `y = 2x − 1` plus noise (std 0.3). The gradients of MSE, by the chain rule you just practised:
`dL/dw = mean(2·(pred − y)·x)` and `dL/db = mean(2·(pred − y))`.

In [ ]:
x = torch.linspace(-2, 2, 50)
y = 2 * x - 1 + 0.3 * torch.randn(50)

def train(x, y, lr, steps=60):
    w, b, losses = 0.0, 0.0, []
    for _ in range(steps):
        pred = w * x + b                          # forward
        err = pred - y
        losses.append((err ** 2).mean().item())   # loss
        dw = (2 * err * x).mean().item()          # backward, by hand
        db = (2 * err).mean().item()
        w, b = w - lr * dw, b - lr * db           # update
    return w, b, losses

In [ ]:
#@title 🔮 Predict E1: lr = 0.1, 60 steps, starting from w=0, b=0. Roughly what w, b and final loss do you expect? (Remember the noise.)
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
w, b, losses = train(x, y, lr=0.1)
print(f"w = {w:.3f}   b = {b:.3f}   final loss = {losses[-1]:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].scatter(x, y, s=10); ax[0].plot(x, w * x + b, "r"); ax[0].set_title("fit")
ax[1].plot(losses); ax[1].set_yscale("log"); ax[1].set_title("loss per step"); plt.show()

The loss floor is not 0: it is the noise variance (0.3² ≈ 0.09). No model can predict the noise. Remember this floor whenever a paper's loss "plateaus".

In [ ]:
#@title 🔮 Predict E2: Learning rates 0.01, 0.1, 0.5, 0.9, 1.1: which converge in 60 steps, which crawl, which diverge?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E2", prediction, confidence)

In [ ]:
for lr in [0.01, 0.1, 0.5, 0.9, 1.1]:
    w, b, losses = train(x, y, lr)
    plt.plot(losses, label=f"lr={lr}")
    print(f"lr={lr:<5} final loss = {losses[-1]:.4g}")
plt.yscale("log"); plt.ylim(1e-2, 1e6); plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.show()
print("steepest curvature = 2*mean(x^2) =", (2 * (x**2).mean()).item(), "-> stable only while lr < 2/curvature =", (1 / (x**2).mean()).item())

In [ ]:
#@title ✍️ E2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E2", what_happened, surprise)

In [ ]:
#@title 🔮 Predict E3: Same data, but x is recorded in different units: x_mm = 10·x. With the 'safe' lr = 0.1, what happens? What is the fix?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E3", prediction, confidence)

In [ ]:
x_mm = 10 * x
for lr in [0.1, 0.01, 0.005]:
    w, b, losses = train(x_mm, y, lr)
    print(f"lr={lr:<6} final loss = {losses[-1]:.4g}")
print("stable only while lr <", (1 / (x_mm**2).mean()).item())

In [ ]:
#@title ✍️ E3: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E3", what_happened, surprise)

Units changed the loss surface into a **ravine** 100× steeper along `w`. Nothing about the problem got harder; only its scale did. This is why everyone normalises inputs, and why week 3 introduces optimizers that adapt the step per parameter. **Now let autograd do the backward pass:**

In [ ]:
w = torch.zeros((), requires_grad=True); b = torch.zeros((), requires_grad=True)
for step in range(60):
    loss = ((w * x + b - y) ** 2).mean()     # forward + loss
    loss.backward()                          # backward: fills w.grad, b.grad
    with torch.no_grad():                    # the update itself is not part of the graph
        w -= 0.1 * w.grad; b -= 0.1 * b.grad
        w.grad.zero_(); b.grad.zero_()       # gradients accumulate: reset them
print(f"w = {w.item():.3f}   b = {b.item():.3f}   loss = {loss.item():.4f}   (same as your hand-written version?)")

Those five lines inside the loop are **the** training loop. GPT-scale training is the same loop with a bigger model, a fancier update rule, and more data.

---
## Part F · Stretch: a tiny autograd engine (25 min)

**Words used in this part.** *computation graph*: each `Value` remembers its parents. *local derivative*: each operation knows only its own. *topological order*: process a node only after everything that depends on it.

This is the seed of next week (micrograd). Read it, run it, then add `__sub__` or `__pow__` yourself.

In [ ]:
class Value:
    def __init__(self, data, parents=()):
        self.data, self.grad, self.parents, self._backward = data, 0.0, parents, lambda: None
    def __add__(self, other):
        out = Value(self.data + other.data, (self, other))
        def _backward():                       # add copies
            self.grad += out.grad; other.grad += out.grad
        out._backward = _backward; return out
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other))
        def _backward():                       # multiply swaps
            self.grad += other.data * out.grad; other.grad += self.data * out.grad
        out._backward = _backward; return out
    def backward(self):
        order, seen = [], set()
        def visit(v):
            if v not in seen:
                seen.add(v); [visit(p) for p in v.parents]; order.append(v)
        visit(self); self.grad = 1.0           # every backward pass starts with a 1
        for v in reversed(order): v._backward()

In [ ]:
#@title 🔮 Predict F1: Using Value: a=1.5, b=4, c=-2, e = a*b + c, L = e*e. Will a.grad, b.grad, c.grad match Part D (32, 12, 8)? Why does the code use += and not = ?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
a, b, c = Value(1.5), Value(4.0), Value(-2.0)
e = a * b + c
L = e * e                      # e is used twice: its gradient must ADD from both uses
L.backward()
print("a.grad, b.grad, c.grad =", a.grad, b.grad, c.grad)

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `core-calc`, and mark the biggest surprise.
Then on the laptop:

```
python track.py done week-01/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 2
```

In [ ]:
reveal()